In [ ]:
import pickle
import numpy as np
from typing import Dict, List

# Load model artifacts
_MODEL_CACHE = None
_SCALER_CACHE = None
_ENCODER_CACHE = None

def _load_model_artifacts():
    global _MODEL_CACHE, _SCALER_CACHE, _ENCODER_CACHE
    
    if _MODEL_CACHE is not None:
        return _MODEL_CACHE, _SCALER_CACHE, _ENCODER_CACHE
    
    try:
        with open('src/models/failure_predictor.pkl', 'rb') as f:
            _MODEL_CACHE = pickle.load(f)
        with open('src/models/feature_scaler.pkl', 'rb') as f:
            _SCALER_CACHE = pickle.load(f)
        with open('src/models/type_encoder.pkl', 'rb') as f:
            _ENCODER_CACHE = pickle.load(f)
    except Exception as e:
        return None, None, None
    
    return _MODEL_CACHE, _SCALER_CACHE, _ENCODER_CACHE


def predict_vehicle_failure_risk(vehicle: Dict) -> Dict:
    """
    Use ML model to predict vehicle failure risk.
    Falls back to rule-based scoring if ML model unavailable.
    """
    model, scaler, encoder = _load_model_artifacts()
    
    if model is None:
        return None
    
    try:
        # Map vehicle telemetry to ML features (all 11 features)
        features = {
            'Type': vehicle.get('vehicle_type', 'M'),
            'Air temperature [K]': float(vehicle.get('engine_temperature', 298)) + 273,
            'Process temperature [K]': float(vehicle.get('engine_temperature', 298)) + 285,
            'Rotational speed [rpm]': float(vehicle.get('mileage', 0)) / 100,
            'Torque [Nm]': float(vehicle.get('brake_wear', 0)) * 0.6,
            'Tool wear [min]': float(vehicle.get('days_since_last_service', 0)),
            'TWF': 0,  # Tool Wear Failure
            'HDF': 0,  # Heat Dissipation Failure
            'PWF': 0,  # Power Failure
            'OSF': 0,  # Overstrain Failure
            'RNF': 0   # Random Nonfatal Failure
        }
        
        feature_cols = ['Type', 'Air temperature [K]', 'Process temperature [K]', 
                       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
                       'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
        
        feature_values = []
        for col in feature_cols:
            if col == 'Type':
                val = encoder.transform([features[col]])[0]
            else:
                val = float(features[col])
            feature_values.append(val)
        
        X_input = np.array([feature_values]).reshape(1, -1)
        X_scaled = scaler.transform(X_input)
        
        prob = model.predict_proba(X_scaled)[0, 1]
        pred = model.predict(X_scaled)[0]
        conf = float(max(model.predict_proba(X_scaled)[0]))
        
        # Map probability to risk level
        if prob < 0.3:
            risk_level = 'Low'
        elif prob < 0.6:
            risk_level = 'Medium'
        elif prob < 0.8:
            risk_level = 'High'
        else:
            risk_level = 'Critical'
        
        return {
            'vehicle_id': vehicle.get('vehicle_id', 'unknown'),
            'risk_level': risk_level,
            'ml_probability': float(prob),
            'predicted_failure': int(pred),
            'confidence': conf,
            'analysis_method': 'machine_learning',
            'key_issues': [
                f'ML Model: {risk_level} failure risk (prob={prob:.2f})',
                f'Based on operational parameters and historical failure patterns'
            ] if prob > 0.3 else ['Normal operation expected']
        }
    except Exception as e:
        return None


def analyze_risks_ml(vehicles: List[Dict]) -> List[Dict]:
    """
    Analyze fleet risks using both ML model and deterministic scoring.
    """
    results = []
    
    for vehicle in vehicles:
        # Try ML prediction first
        ml_result = predict_vehicle_failure_risk(vehicle)
        
        if ml_result:
            result = ml_result
        else:
            # Fallback to rule-based scoring
            result = _score_vehicle(vehicle)
            result['analysis_method'] = 'rule_based'
        
        results.append(result)
    
    return results


def _score_vehicle(vehicle: Dict) -> Dict:
    """Deterministic rule-based vehicle scoring (fallback)."""
    score = 0
    issues: List[str] = []

    mileage = float(vehicle.get("mileage", 0))
    engine_temp = float(vehicle.get("engine_temperature", 0))
    oil_quality = float(vehicle.get("oil_quality", 100))
    brake_wear = float(vehicle.get("brake_wear", 0))
    battery_health = float(vehicle.get("battery_health", 100))
    days_since_service = float(vehicle.get("days_since_last_service", 0))

    if mileage > 150000:
        score += 2
        issues.append("High mileage vehicle")
    elif mileage > 90000:
        score += 1

    if engine_temp > 105:
        score += 3
        issues.append("Engine running above safe temperature")
    elif engine_temp > 95:
        score += 1

    if oil_quality < 35:
        score += 2
        issues.append("Poor oil quality indicates immediate service need")
    elif oil_quality < 55:
        score += 1

    if brake_wear > 80:
        score += 3
        issues.append("Brake system near wear limit")
    elif brake_wear > 60:
        score += 2

    if battery_health < 30:
        score += 2
        issues.append("Battery health is critically low")
    elif battery_health < 50:
        score += 1

    if days_since_service > 220:
        score += 2
        issues.append("Service overdue by schedule")
    elif days_since_service > 150:
        score += 1

    if score >= 9:
        risk_level = "Critical"
    elif score >= 6:
        risk_level = "High"
    elif score >= 3:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    confidence = min(0.98, 0.5 + (score / 14))

    return {
        "vehicle_id": str(vehicle.get("vehicle_id", "unknown")),
        "risk_level": risk_level,
        "key_issues": issues[:4] or ["No immediate critical issues detected"],
        "confidence": round(confidence, 2),
    }
